# Ungraded Lab 2: LLM Calls and Crafting Simple Augmented Prompts


Welcome to **LLM Calls and Crafting Simple Augmented Prompts**. In this lab, you'll get hands-on practice with using two essential functions that let you interact with Large Language Models (LLMs). These functions help you both send single prompts to an LLM, and have a back-and-forth conversation. The main aim is to show you how to add extra information to your prompts, making them more detailed and useful. This added context helps the model give you better and more precise responses.

In this lab, you'll learn:

- How to set up and send questions to an LLM for both single questions and conversations.
- How to use additional data to make your prompts richer, improving the model's replies.


# Table of Contents
- [ 1 - Understanding the functions to call LLMs](#1)
  - [ 1.1 `generate_with_single_input`](#1-1)
  - [ 1.2 `generate_with_multiple_input`](#1-2)
- [ 2 - Integrating Data into an LLM Prompt](#2)
  - [ 2.1 Understanding the data structure](#2-1)
  - [ 2.2 Creating the Prompt](#2-2)

In [1]:
from utils import (
    generate_with_multiple_input,
    generate_with_single_input,
    get_proxy_headers,
    get_proxy_url,
    get_together_key
)

<a id='1'></a>
## 1 - Understanding the functions to call LLMs

In this section you will explore the one function that will be used to call LLMs in this course. This function calls the [together.ai](https://www.together.ai/) API to call the models. Here in the Coursera environment some steps in reaching the Together API are handled on your behalf via a proxy server, so if you try to run this notebook outside the Coursera environment it won't work right away. With small adjustments, however, you can pass an optional parameter with a together.ai API key, which will allow you to run these notebooks on your local machine.

<a id='1-1'></a>
### 1.1 `generate_with_single_input`

This function allows you to generate text from a language model based on a single input prompt. For now, let's just focus on a simple call with only a few basic parameters. You will explore different parameters to call an LLM and their impact on the output in Module 4. Here's the parameters you'll have access to for now.

#### Parameters:

- `prompt` (str): The input text prompt you want to send to the language model.
- `max_tokens` (int): Maximum tokens to generate in the response.
- `model` (str): The model name. Default is `"google/gemma-3n-E4B-it"`.
- The request payload disables reasoning with `reasoning={"enabled": False}` to avoid generating unnecessary reasoning tokens.
- `together_api_key`: An optional API key for authentication; defaults to `None`. If `None` you will use our proxy, otherwise a direct call to together.ai will be performed with the provided API key.

```python
from together import Together

client = Together()

response = client.chat.completions.create(
    model="Qwen/Qwen3.5-9B",
    reasoning={"enabled": False},
    messages=[
        {
            "role": "user",
            "content": "What are some fun things to do in New York?",
        }
    ],
)

print(response.model_dump()['choices'][-1]['message']['role'])

# print(response.choices[0].message.content)
```

In [2]:
# Example call
output = generate_with_single_input(
    prompt="What is the capital of France?"
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: The capital of France is **Paris**.

Located in the north-central part of the country along the Seine River, Paris is France's largest city and a major global center for art, fashion, culture, finance, and diplomacy. It has served as the nation's capital since the Middle Ages.


In [3]:
# Example call
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: The captain of the France national team when they won the 2018 FIFA World Cup was **Hugo Lloris**.

As the long-standing goalkeeper for the national side, he wore the captain's armband throughout the tournament, including during the final match against Croatia on July 15, 2018, where France secured victory on a penalty shootout after a 1-1 draw.


### 1.3 Integration with OpenAI library

[Together.ai](together.ai) endpoints are [OpenAI compatible](https://docs.together.ai/docs/openai-api-compatibility) so you can use the [OpenAI library](https://github.com/openai/openai-python) to make the calls. In this section you will explore how to do it.  

In [4]:
from openai import OpenAI

In [5]:
base_url = get_proxy_url() # If using together endpoint, add it here https://api.together.xyz/

client = OpenAI(
    api_key = get_together_key(), # Set any as our proxy does not use it. Set the together api key if using the together endpoint.
    base_url=base_url,
)

In [6]:
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

In [7]:
response = client.chat.completions.create(messages = messages, model ="Qwen/Qwen3.5-9B", extra_body={
        "reasoning": False
    })

In [10]:
print(response)

ChatCompletion(id='ofciT3m-6Ng1vN-9ee2bdd64c713eb1', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='Thinking Process:\n\n1.  **Analyze the Request:** The user is asking for the captain of the team that won the 2018 FIFA World Cup. Based on the previous turn, I know France won the 2018 World Cup, so the question is specifically about the captain of the French national team during that tournament.\n\n2.  **Retrieve Knowledge:** Access knowledge about the 2018 FIFA World Cup.\n    *   Tournament: 2018 FIFA World Cup.\n    *   Host: Russia.\n    *   Winner: France.\n    *   Key Player/Team Captain: The captain of the French national football team during the 2018 World Cup.\n    *   Specific Name: Kylian Mbappé was a star, but was he captain? No.\n    *   Specific Name: Hansi Flick was a coach (no, that\'s FC Bayern). Didier D

<a id='2'></a>
## 2 - Integrating Data into an LLM Prompt

In this section, you will learn how to effectively incorporate data into a prompt before passing it to a Large Language Model (LLM). We will work with a small dataset consisting of JSON files that contain information about houses. It will help you understand how to augment prompts in the context of RAG.

<a id='2-1'></a>
### 2.1 Understanding the data structure

Let's have a quick look in the data structure. It is a tiny dataset of houses. A list containing one dictionary per house.

In [8]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

<a id='2-2'></a>
### 2.2 Creating the Prompt

Let's begin by constructing the prompt. The first step is to design a layout for the data.

In [9]:
# First, let's create a layout for the houses

def house_info_layout(houses):
    # Create an empty string
    layout = ''
    # Iterate over the houses
    for house in houses:
        # For each house, append the information to the string using f-strings
        # The following way using brackets is a good way to make the code readable as in each line you can start a new f-string that will appended to the previous one
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
            f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
            f"{house['square_feet']} sq ft area, priced at ${house['price']}, "
            f"built in {house['year_built']}.\n") # Don't forget the new line character at the end!
    return layout

In [11]:
# Check the layout
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.



In [12]:
def generate_prompt(query, houses):
    # The code made above is modular enough to accept any list of houses, so you could also choose a subset of the dataset.
    # This might be useful in a more complex context where you want to give only some information to the LLM and not the entire data
    houses_layout = house_info_layout(houses)
    # Create a hard-coded prompt. You can use three double quotes (") in this cases, so you don't need to worry too much about using single or double quotes and breaking the code
    PROMPT = f"""
Use the following houses information to answer users queries.
{houses_layout}
Query: {query}    
             """
    return PROMPT

In [13]:
print(generate_prompt("What is the most expensive house?", houses = house_data))


Use the following houses information to answer users queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.

Query: What is the most expensive house?    
             


In [14]:
query = "What is the most expensive house? And the bigger one?"
# Asking without the augmented prompt, let's pass the role as user
query_without_house_info = generate_with_single_input(prompt = query, role = 'user')
# With house info, given the prompt structuer, let's pass the role as assistant
enhanced_query = generate_prompt(query, houses = house_data)
query_with_house_info = generate_with_single_input(prompt = enhanced_query, role = 'assistant')

In [15]:
query_with_house_info

{'role': 'assistant',
 'content': 'Based on the housing information provided:\n\n*   **Most Expensive House:** The house at **456 Elm Avenue, Shelbyville, TN 37160** is the most expensive, priced at **$320,000**.\n*   **Bigger House:** The house at **456 Elm Avenue, Shelbyville, TN 37160** is also the bigger one, with a total area of **2,500 sq ft**.\n\n*(Note: The house at 123 Maple Street costs $230,000 and has an area of 1,500 sq ft, making the Elm Avenue property superior in both metrics.)*'}

In [18]:
print(query_with_house_info['content'])

Based on the housing information provided:

*   **Most Expensive House:** The house at **456 Elm Avenue, Shelbyville, TN 37160** is the most expensive, priced at **$320,000**.
*   **Bigger House:** The house at **456 Elm Avenue, Shelbyville, TN 37160** is also the bigger one, with a total area of **2,500 sq ft**.

*(Note: The house at 123 Maple Street costs $230,000 and has an area of 1,500 sq ft, making the Elm Avenue property superior in both metrics.)*
